# Goal
Practice creating metadata by applying what you learned in class to a dataset of your choice.

# Task
Select any publicly available dataset, for example from Kaggle, Zenodo, or Figshare, and create the following:
- DataCite metadata in XML format
- schema.org metadata in JSON-LD format using @type: Dataset


## Dataset

**Train Stations in Europe** by Martin Henze (headsortails), published on Kaggle (2023).
Source: https://www.kaggle.com/datasets/headsortails/train-stations-in-europe

## Part 1: Dataset Inspection

### Step 1: Import libraries and load data

`low_memory=False` is used to ensure pandas reads the entire file before inferring column
types, which avoids unreliable type guesses when a column contains mixed content.

In [1]:
import pandas as pd

df = pd.read_csv("../../data/train_stations_europe.csv", low_memory=False)

df.head()

,id,name,name_norm,uic,latitude,longitude,parent_station_id,country,time_zone,is_city,is_main_station,is_airport,entur_id,entur_is_enabled,trenord_id,cercanias_id,cercanias_hub_id,cercanias_is_enabled
0,1,Château-Arnoux – St-Auban,Chateau-Arnoux - St-Auban,NaN,44.081790,6.001625,NaN,FR,Europe/Paris,True,False,False,NaN,False,NaN,NaN,NaN,False
1,2,Château-Arnoux – St-Auban,Chateau-Arnoux - St-Auban,8775123.0,44.061565,5.997373,1.0,FR,Europe/Paris,False,True,False,NaN,False,NaN,NaN,NaN,False
2,3,Château-Arnoux Mairie,Chateau-Arnoux Mairie,8775122.0,44.063863,6.011248,1.0,FR,Europe/Paris,False,False,False,NaN,False,NaN,NaN,NaN,False
3,4,Digne-les-Bains,Digne-les-Bains,NaN,44.350000,6.350000,NaN,FR,Europe/Paris,True,False,False,NaN,False,NaN,NaN,NaN,False
4,6,Digne-les-Bains,Digne-les-Bains,8775149.0,44.088710,6.222982,4.0,FR,Europe/Paris,False,True,False,NaN,False,NaN,NaN,NaN,False


### Step 2 — Basic structure

In [2]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)

Shape: (70524, 18)

Columns: Index(['id', 'name', 'name_norm', 'uic', 'latitude', 'longitude',
       'parent_station_id', 'country', 'time_zone', 'is_city',
       'is_main_station', 'is_airport', 'entur_id', 'entur_is_enabled',
       'trenord_id', 'cercanias_id', 'cercanias_hub_id',
       'cercanias_is_enabled'],
      dtype='str')


### Step 3: Data types

In [3]:
df.dtypes.to_frame("dtype")

,dtype
id,int64
name,str
name_norm,str
uic,float64
latitude,float64
longitude,float64
parent_station_id,float64
country,str
time_zone,str
is_city,bool


The boolean columns are read correctly. The `uic` column is `float64` rather than an integer
type as pandas promotes integer columns to float when `NaN` values are present, which is the
case here. The same applies to the three vendor ID columns (`trenord_id`, `cercanias_id`,
`cercanias_hub_id`).


### Step 4: Missing values

In [4]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_pct"] = (missing["missing_count"] / len(df) * 100).round(1)
missing[missing["missing_count"] > 0]

,missing_count,missing_pct
name,1,0.0
name_norm,1,0.0
uic,46788,66.3
latitude,1831,2.6
longitude,1831,2.6
parent_station_id,65229,92.5
entur_id,69888,99.1
trenord_id,70524,100.0
cercanias_id,70524,100.0
cercanias_hub_id,70524,100.0


There are two distinct categories of missing data here.

The first is substantive. UIC codes are absent for approximately **66% of rows**. The dataset
description acknowledges this but offers no explanation. Coordinates are missing for roughly
**2.6% of rows** — again acknowledged, but undocumented in terms of why or which stations are
affected. Both gaps directly limit how useful this dataset is for spatial or network analysis
without additional cleaning.

The second is structural. The vendor-specific columns (`trenord_id`, `cercanias_id`,
`cercanias_hub_id`) are null for every single row in this file. The dataset creator explicitly
chose to exclude most operator-specific columns from the upstream source, but these three were
retained despite being empty. They add no information and no documentation explains when, if
ever, they would be populated.

In [5]:
### Step 5: Duplicates and geographic coverage

In [6]:
print("Duplicate rows:", df.duplicated().sum())
print("\nCountry distribution (top 10):")
df["country"].value_counts().head(10).to_frame("station_count")

Duplicate rows: 0

Country distribution (top 10):


,station_count
country,
CH,22345
DE,14117
FR,7388
SE,5767
ES,5241
IT,4981
GB,2788
AT,1710
NO,856


There are **no duplicate rows**, which is a straightforward quality positive.

The dataset covers **43 country codes**, though the distribution is heavily skewed with
Switzerland alone accounting for roughly 32% of all rows. More importantly, the dataset
description states the file contains "36,000+ stations", but the actual row count is
**70,524**. This is Version 23 of the dataset (updated November 2025), yet the written
description has not been updated to reflect the current row count. The version history exists
in Kaggle's interface but is not embedded in the file itself. A user working only with the
CSV has no way of knowing which version they have or what changed between versions.

The Kaggle page provides a title and a description. No formally structured keywords are
defined. The creator is identifiable by name and Kaggle username but no institutional
affiliation or persistent researcher identifier such as an ORCID is provided.

### Step 7: Structural metadata

Structural metadata explains how the dataset is organised, including what each row and column
represents.

In [7]:
structural_metadata = {
    "row_unit": "One row represents one railway station.",
    "columns": {
        "id":                   "Numeric internal unique identifier. Primary key.",
        "name":                 "Station name as locally known, including accents and special characters.",
        "name_norm":            "Normalised version of name in Latin-ASCII character space.",
        "uic":                  "UIC (International Union of Railways) station code. Absent for approximately 66% of rows.",
        "latitude":             "Station latitude in decimal degrees. Absent for approximately 2.6% of rows.",
        "longitude":            "Station longitude in decimal degrees. Absent for approximately 2.6% of rows.",
        "parent_station_id":    "ID of a parent meta-station if applicable. Absent for approximately 92% of rows.",
        "country":              "ISO 3166-1 alpha-2 country code.",
        "time_zone":            "IANA time zone string (e.g. Europe/Paris).",
        "is_city":              "Flagged as unreliable in the source dataset.",
        "is_main_station":      "Flagged as unreliable in the source dataset.",
        "is_airport":           "TRUE if the station is at or connected to an airport.",
        "entur_id":             "Norwegian Entur platform identifier. Absent for approximately 99% of rows.",
        "entur_is_enabled":     "Boolean flag for whether the station is active in the Entur system.",
        "trenord_id":           "Trenord (Italian regional rail) identifier. Absent for all rows in this file.",
        "cercanias_id":         "Cercanias (Spanish suburban rail) identifier. Absent for all rows in this file.",
        "cercanias_hub_id":     "Cercanias hub identifier. Absent for all rows in this file.",
        "cercanias_is_enabled": "Boolean flag for whether the station is active in the Cercanias system.",
    }
}

pd.DataFrame([
    {"Column": k, "Description": v}
    for k, v in structural_metadata["columns"].items()
])

,Column,Description
0,id,Numeric internal unique identifier. Primary key.
1,name,"Station name as locally known, including accen..."
2,name_norm,Normalised version of name in Latin-ASCII char...
3,uic,UIC (International Union of Railways) station ...
4,latitude,Station latitude in decimal degrees. Absent fo...
5,longitude,Station longitude in decimal degrees. Absent f...
6,parent_station_id,ID of a parent meta-station if applicable. Abs...
7,country,ISO 3166-1 alpha-2 country code.
8,time_zone,IANA time zone string (e.g. Europe/Paris).
9,is_city,Flagged as unreliable in the source dataset.


The column descriptions above are drawn from the Kaggle dataset page, which in turn draws
from the upstream GitHub repository. Two columns, `is_city` and `is_main_station`, are
explicitly flagged as unreliable in the source. The three Cercanias columns are entirely
null in this version of the file.

## Part 2: DataCite Metadata

### Step 1: Define the metadata dictionary

All metadata values are defined in a single dictionary. This same dictionary will be reused
in Part 3 for the schema.org record.

The dataset has no DOI. The Kaggle URL is used as the identifier with type `URL`. This is
noted explicitly as a limitation.

In [8]:
metadata = {
    "identifier":           "https://www.kaggle.com/datasets/headsortails/train-stations-in-europe",
    "identifier_type":      "URL",
    "creator_name":         "Henze, Martin",
    "creator_type":         "Personal",
    "title":                "Train Stations in Europe",
    "publisher":            "Kaggle",
    "publication_year":     "2020",
    "resource_type":        "Dataset",
    "description":          (
                            "Names, coordinates, and basic properties of over 70,000 railway "
                            "stations in Europe and adjacent regions. Derived from the "
                            "trainline-eu/stations repository maintained by Trainline EU."
    ),
    "subjects":             [
                            "railway stations",
                            "train stations",
                            "Europe",
                            "coordinates",
                            "geospatial",
                            "transportation",
                            "UIC codes",
    ],
    "license":              "https://opendatacommons.org/licenses/odbl/1-0/",
    "access_url":           "https://www.kaggle.com/datasets/headsortails/train-stations-in-europe",
    "language":             "en",
    "version":              "23",
}

### Step 2: Generate the DataCite XML

The XML structure follows the DataCite Metadata Schema kernel-4. The namespace declaration
and schema location are taken directly from the course example file
`climate_dataset_datacite.xml`. Different versions of the DataCite schema exist; this record
uses the same version as the course examples.

In [9]:
import xml.etree.ElementTree as ET
from xml.dom import minidom

def build_datacite_xml(meta):
    root = ET.Element("resource")
    root.set("xmlns", "http://datacite.org/schema/kernel-4")
    root.set("xmlns:xsi", "http://www.w3.org/2001/XMLSchema-instance")
    root.set(
        "xsi:schemaLocation",
        "http://datacite.org/schema/kernel-4 "
        "http://schema.datacite.org/meta/kernel-4/metadata.xsd"
    )

    identifier = ET.SubElement(root, "identifier")
    identifier.set("identifierType", meta["identifier_type"])
    identifier.text = meta["identifier"]

    creators = ET.SubElement(root, "creators")
    creator = ET.SubElement(creators, "creator")
    creator_name = ET.SubElement(creator, "creatorName")
    creator_name.set("nameType", meta["creator_type"])
    creator_name.text = meta["creator_name"]

    titles = ET.SubElement(root, "titles")
    title = ET.SubElement(titles, "title")
    title.text = meta["title"]

    publisher = ET.SubElement(root, "publisher")
    publisher.text = meta["publisher"]

    pub_year = ET.SubElement(root, "publicationYear")
    pub_year.text = meta["publication_year"]

    resource_type = ET.SubElement(root, "resourceType")
    resource_type.set("resourceTypeGeneral", meta["resource_type"])
    resource_type.text = meta["resource_type"]

    descriptions = ET.SubElement(root, "descriptions")
    description = ET.SubElement(descriptions, "description")
    description.set("descriptionType", "Abstract")
    description.text = meta["description"]

    subjects = ET.SubElement(root, "subjects")
    for kw in meta["subjects"]:
        subject = ET.SubElement(subjects, "subject")
        subject.text = kw

    rights_list = ET.SubElement(root, "rightsList")
    rights = ET.SubElement(rights_list, "rights")
    rights.set("rightsURI", meta["license"])
    rights.text = "Open Database License (ODbL) v1.0"

    language = ET.SubElement(root, "language")
    language.text = meta["language"]

    version = ET.SubElement(root, "version")
    version.text = meta["version"]

    raw_xml = ET.tostring(root, encoding="unicode")
    parsed = minidom.parseString(raw_xml)
    return parsed.toprettyxml(indent="  ")


datacite_xml = build_datacite_xml(metadata)
print(datacite_xml)

<?xml version="1.0" ?>
<resource xmlns="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4/metadata.xsd">
  <identifier identifierType="URL">https://www.kaggle.com/datasets/headsortails/train-stations-in-europe</identifier>
  <creators>
    <creator>
      <creatorName nameType="Personal">Henze, Martin</creatorName>
    </creator>
  </creators>
  <titles>
    <title>Train Stations in Europe</title>
  </titles>
  <publisher>Kaggle</publisher>
  <publicationYear>2020</publicationYear>
  <resourceType resourceTypeGeneral="Dataset">Dataset</resourceType>
  <descriptions>
    <description descriptionType="Abstract">Names, coordinates, and basic properties of over 70,000 railway stations in Europe and adjacent regions. Derived from the trainline-eu/stations repository maintained by Trainline EU.</description>
  </descriptions>
  <subjects>
    <subject>rail

### Step 3: Save the XML file

In [10]:
with open("train_stations_europe_datacite.xml", "w", encoding="utf-8") as f:
    f.write(datacite_xml)

print("Saved: train_stations_europe_datacite.xml")

Saved: train_stations_europe_datacite.xml


The record uses a URL identifier rather than a DOI, as no DOI has been assigned to this
dataset. The `creator_name` follows the DataCite convention of family name first. The
subjects field is constructed from the informal Kaggle tags combined with terms drawn from
the column content, as no controlled vocabulary keywords are provided in the original record.
The version corresponds to Kaggle's version numbering (Version 23, updated November 2025).

## Part 3: schema.org Metadata

### Step 1: Generate the schema.org JSON-LD

The record uses `@type: Dataset` as required by the task. The metadata dictionary defined in
Part 2 is reused directly. The JSON-LD structure follows the course example file
`climate_dataset_schemaorg.jsonld`. The creator is typed as `Person` rather than
`Organization` as Martin Henze is an individual contributor.

In [11]:
import json

schemaorg = {
    "@context":     "https://schema.org",
    "@type":        "Dataset",
    "name":         metadata["title"],
    "description":  metadata["description"],
    "creator": [
        {
            "@type": "Person",
            "name":  "Martin Henze",
        }
    ],
    "publisher": {
        "@type": "Organization",
        "name":  metadata["publisher"],
    },
    "identifier":       metadata["identifier"],
    "keywords":         metadata["subjects"],
    "license":          metadata["license"],
    "version":          metadata["version"],
    "inLanguage":       metadata["language"],
    "distribution": [
        {
            "@type":            "DataDownload",
            "encodingFormat":   "text/csv",
            "contentUrl":       metadata["access_url"],
            "name":             "train_stations_europe.csv",
        }
    ],
}

print(json.dumps(schemaorg, indent=2))

{
  "@context": "https://schema.org",
  "@type": "Dataset",
  "name": "Train Stations in Europe",
  "description": "Names, coordinates, and basic properties of over 70,000 railway stations in Europe and adjacent regions. Derived from the trainline-eu/stations repository maintained by Trainline EU.",
  "creator": [
    {
      "@type": "Person",
      "name": "Martin Henze"
    }
  ],
  "publisher": {
    "@type": "Organization",
    "name": "Kaggle"
  },
  "identifier": "https://www.kaggle.com/datasets/headsortails/train-stations-in-europe",
  "keywords": [
    "railway stations",
    "train stations",
    "Europe",
    "coordinates",
    "geospatial",
    "transportation",
    "UIC codes"
  ],
  "license": "https://opendatacommons.org/licenses/odbl/1-0/",
  "version": "23",
  "inLanguage": "en",
  "distribution": [
    {
      "@type": "DataDownload",
      "encodingFormat": "text/csv",
      "contentUrl": "https://www.kaggle.com/datasets/headsortails/train-stations-in-europe",
      

### Step 2: Save the JSON-LD file

In [12]:
with open("train_stations_europe_schemaorg.jsonld", "w", encoding="utf-8") as f:
    json.dump(schemaorg, f, indent=2)

print("Saved: train_stations_europe_schemaorg.jsonld")

Saved: train_stations_europe_schemaorg.jsonld


The `keywords` and `identifier` fields are drawn from the same dictionary used to generate
the DataCite record. The `distribution` block points to the Kaggle dataset page as the
access URL, since no direct stable download link exists outside the platform.